# 🚀 TASK 6.3-C: CHIẾN DỊCH HUẤN LUYỆN THÍCH ỨNG ĐA MIỀN TRÊN GOOGLE COLAB
### Huấn luyện 3 nhánh ứng viên M3 (+Data), M4 (+Distillation), M5 (+LoRA) — Milestone 6

> **Mã nhiệm vụ**: `Task 6.3-C` *(Milestone 6 - Tuần 6)*  
> **Người chủ trì**: **Thành viên C (Training & Evaluation Lead)**  
> **Người phối hợp**: **Thành viên B (Architecture Lead)**, **Thành viên A (Data & Infra Lead)**  
> **Môi trường thực thi**: Google Colab (GPU Tesla T4 15GB VRAM hoặc Colab Pro A100/L4)  
> **Kế thừa hạ tầng**: Kho dữ liệu Google Drive 5TB; Trọng số mỏ neo Plan A từ Task 4.3; Khóa cố định 18 tập test Task 5.1  

---
### 📌 TIÊU CHUẨN NGHIỆM THU (DEFINITION OF DONE - DoD):
- [ ] Nạp 100% dữ liệu ảnh thật từ `diffusion_staging.tar` trên Google Drive 5TB, giải nén sang SSD NVMe Colab.
- [ ] Vượt qua kiểm thử khớp nối Smoke Gate Milestone 6 trước khi khởi động huấn luyện 5 epochs.
- [ ] Cố định điều kiện đối chứng cho cả 3 nhánh: cùng random seed (42), cùng tập dữ liệu, cùng trọng số mỏ neo Plan A, cùng learning rate schedule (5 epochs).
- [ ] Tuân thủ chính sách đĩa: Chỉ lưu `crossdomain_model_best.pth` và `checkpoint_latest.pth`, không lưu dư thừa từng epoch.
- [ ] Thu thập đầy đủ bản ghi tiến trình qua từng epoch vào `DATASET/logs/m6_training_history.csv`.
- [ ] Tự động sao lưu các checkpoint tuyển chọn sang Google Drive 5TB sau khi hoàn thành.


## 1. Kết Nối Google Drive 5TB & Thiết Lập Không Gian Làm Việc
* Mount Google Drive vào `/content/drive`.
* Đồng bộ mã nguồn từ GitHub repository nhánh `hoang`.
* Kiểm tra thông số phần cứng GPU khả dụng.


In [ ]:
# ==============================================================================
# 1. KET NOI GOOGLE DRIVE VA THIET LAP KHONG GIAN LAM VIEC
# ==============================================================================
import os
import sys
import time
import torch

print(f"[*] Phien ban PyTorch: {torch.__version__}")
print(f"[*] CUDA kha dung: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"[*] GPU dang su dung: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("[!] Canh bao: Khong tim thay GPU, script se chay tren CPU (che do dry-run)!")

# Ket noi Google Drive
print("[*] Dang ket noi Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

WORKSPACE = '/content/fatformer-xla'
BRANCH = 'hoang'
REPO_URL = 'https://github.com/LeeVHoangtk3/fatformer-xla.git'

if not os.path.exists(WORKSPACE):
    print(f"[*] Dang clone ma nguon tu {REPO_URL} (nhanh {BRANCH})...")
    os.system(f"git clone -b {BRANCH} {REPO_URL} {WORKSPACE}")
else:
    print(f"[*] Dong bo commit moi tu nhanh {BRANCH}...")
    os.system(f"git -C {WORKSPACE} fetch origin {BRANCH}")
    os.system(f"git -C {WORKSPACE} reset --hard origin/{BRANCH}")

if WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)
os.chdir(WORKSPACE)
print(f"[✓] Thu muc lam viec hien tai: {os.getcwd()}")


## 2. Giải Nén Kho Dữ Liệu Thật Từ Google Drive & Chuẩn Bị Trọng Số Mỏ Neo
* Sao chép và giải nén tệp `diffusion_staging.tar` (1.80 GB, 3.600 ảnh catalyst) từ Google Drive sang SSD NVMe Colab `/content/dataset_local/`.
* Liên kết trọng số Backbone CLIP (`ViT-L-14.pt`) và trọng số khởi tạo Plan A (`fatformer_plan_a_best_weights.pth`).
* Tự động thiết lập tệp kê khai `DATASET/manifests/data_manifest.json` trỏ trực tiếp đến 100% ảnh thật trên SSD NVMe.


In [ ]:
# ==============================================================================
# 2. GIAI NEN KHO DU LIEU THAT VA CHUAN BI TRONG SO MO NEO
# ==============================================================================
import os
import shutil
import glob
import json
import time
import random

DRIVE_HUB = '/content/drive/MyDrive/Fatformer'
LOCAL_DATA = '/content/dataset_local'
os.makedirs(LOCAL_DATA, exist_ok=True)

# 1. Giai nen tap du lieu diffusion_staging.tar
tar_candidates = [
    os.path.join(DRIVE_HUB, 'datasets', 'diffusion_staging.tar'),
    os.path.join(DRIVE_HUB, 'datasets', 'diffusion_staging', 'diffusion_staging.tar'),
    '/content/drive/MyDrive/Fatformer/datasets/diffusion_staging.tar',
    '/content/diffusion_staging.tar'
]
tar_path = None
for tc in tar_candidates:
    if os.path.exists(tc):
        tar_path = tc
        break

if tar_path:
    size_mb = os.path.getsize(tar_path) / (1024 ** 2)
    print(f"[*] Phat hien goi du lieu nen: {tar_path} ({size_mb:.1f} MB)")
    t0 = time.time()
    os.system(f'tar -xf "{tar_path}" -C "{LOCAL_DATA}"')
    print(f"[✓] Giai nen hoan tat trong {time.time() - t0:.1f}s sang {LOCAL_DATA}")
else:
    print("[!] Canh bao: Khong tim thay diffusion_staging.tar tren Google Drive! Vui long kiem tra lai.")

# 2. Lien ket trong so Backbone CLIP ViT-L-14.pt
os.makedirs('/content/fatformer-xla/DATASET/pretrained', exist_ok=True)
os.makedirs('/content/fatformer-xla/DATASET/checkpoints', exist_ok=True)

clip_candidates = [
    os.path.join(DRIVE_HUB, 'pretrained', 'ViT-L-14.pt'),
    os.path.join(DRIVE_HUB, 'checkpoint', 'ViT-L-14.pt'),
    '/content/drive/MyDrive/Fatformer/pretrained/ViT-L-14.pt',
]
for c in clip_candidates:
    if os.path.exists(c):
        for dst in ['/content/fatformer-xla/DATASET/pretrained/ViT-L-14.pt', '/content/fatformer-xla/ViT-L-14.pt']:
            if not os.path.exists(dst):
                shutil.copy2(c, dst)
        print(f"[✓] Da lien ket CLIP Backbone tu: {c}")
        break

# 3. Lien ket trong so khoi tao Plan A
init_candidates = [
    os.path.join(DRIVE_HUB, 'checkpoint', 'fatformer_plan_a_best_weights.pth'),
    os.path.join(DRIVE_HUB, 'checkpoints', 'fatformer_plan_a_best_weights.pth'),
    os.path.join(DRIVE_HUB, 'checkpoint', 'fatformer_plan_a_final.pth'),
    os.path.join(DRIVE_HUB, 'checkpoint', 'fatformer_plan_a_best.pth'),
    '/content/drive/MyDrive/Fatformer/checkpoint/fatformer_plan_a_best_weights.pth',
]
for c in init_candidates:
    if os.path.exists(c):
        dst = '/content/fatformer-xla/DATASET/checkpoints/fatformer_plan_a_best_weights.pth'
        if not os.path.exists(dst):
            shutil.copy2(c, dst)
        print(f"[✓] Da lien ket Checkpoint Plan A tu: {c}")
        break

# 4. Quet toan bo anh thuc te va lap manifest chuan xac 100%
all_images = []
for root, _, files in os.walk(LOCAL_DATA):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
            p = os.path.join(root, f)
            norm = p.replace('\\', '/').lower()
            is_fake = ('fake' in norm or '1_fake' in norm)
            gen = 'real'
            if is_fake:
                for g in ['stable_diffusion', 'sdv15', 'sd_v15', 'midjourney', 'stylegan', 'progan', 'dalle']:
                    if g in norm:
                        gen = g
                        break
                if gen == 'real':
                    gen = os.path.basename(root)
            all_images.append({
                'file_path': p,
                'label': 1 if is_fake else 0,
                'generator': gen,
                'source': 'diffusion_staging'
            })

print(f"[*] Tong so anh thuc te tim thay: {len(all_images)}")
if all_images:
    rng = random.Random(42)
    reals = [x for x in all_images if x['label'] == 0]
    fakes = [x for x in all_images if x['label'] == 1]
    rng.shuffle(reals)
    rng.shuffle(fakes)

    n_val_r = max(1, int(len(reals) * 0.15))
    n_val_f = max(1, int(len(fakes) * 0.15))

    val_list = reals[:n_val_r] + fakes[:n_val_f]
    train_list = reals[n_val_r:] + fakes[n_val_f:]

    for item in train_list:
        item['split'] = 'train'
    for item in val_list:
        item['split'] = 'val'

    manifest_records = train_list + val_list
    manifest_data = {
        'metadata': {
            'task': 'Task 6.3-C Real Dataset Staging',
            'total_samples': len(manifest_records),
            'train_samples': len(train_list),
            'val_samples': len(val_list),
            'real_count': len(reals),
            'fake_count': len(fakes),
        },
        'records': manifest_records
    }

    manifest_path = '/content/fatformer-xla/DATASET/manifests/data_manifest.json'
    os.makedirs(os.path.dirname(manifest_path), exist_ok=True)
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(manifest_data, f, indent=2)
    print(f"[✓] Da lap thanh cong manifest anh that: {manifest_path} (Train: {len(train_list)}, Val: {len(val_list)})")


## 3. Kích Hoạt Kiểm Tra Khớp Nối Cổng Smoke Gate Milestone 6
Thực hiện kiểm tra tự động trước khi bước vào chiến dịch huấn luyện:
* Đảm bảo dữ liệu từ DataLoader nạp mượt mà, phân bổ cân bằng 50/50 Real/Fake.
* Đảm bảo mô hình Teacher đóng băng 100%, không rò rỉ gradient.
* Đảm bảo LoRA Adapter (163.840 tham số trên các tầng MLP của ViT-L/14) nhận gradient thông suốt.
* Đảm bảo hàm tổn thất `CrossDomainTotalLoss` hội tụ ổn định, không xuất hiện NaN/Inf.


In [ ]:
# ==============================================================================
# 3. KICH HOAT KIEM TRA KHOP NOI SMOKE GATE MILESTONE 6
# ==============================================================================
smoke_script = os.path.join(WORKSPACE, 'tools', 'test_smoke_m6.py')
cmd_smoke = f"python -u {smoke_script}"

print(f"[*] Thuc thi Smoke Gate: {cmd_smoke}")
t0 = time.time()
ret_smoke = os.system(cmd_smoke)
elapsed_smoke = time.time() - t0

if ret_smoke != 0:
    raise RuntimeError(f"[X] Smoke Gate Milestone 6 that bai voi exit code: {ret_smoke}!")
else:
    print(f"\n[✓] Smoke Gate Milestone 6 VUOT QUA 100% TIEU CHUAN trong {elapsed_smoke:.2f}s! San sang huan luyen.")


## 4. Huấn Luyện Nhánh M3: Mở Rộng Dữ Liệu Đa Miền (Data Only — 5 Epochs)
* **Cấu hình**: `configs/train_m3_data.yaml`
* **Đặc trưng**: Mở rộng dữ liệu đa bộ sinh với hàm mất mát Cross-Entropy tiêu chuẩn (label smoothing = 0.05).
* **Mục tiêu đối chứng**: Đo đạc mức cải thiện khi chỉ mở rộng phân phối ảnh huấn luyện, chưa áp dụng chống quên tri thức mỏ neo.


In [ ]:
# ==============================================================================
# 4. HUAN LUYEN NHANH M3 — DATA ONLY (5 EPOCHS)
# ==============================================================================
train_script = os.path.join(WORKSPACE, 'src', 'training', 'train_crossdomain.py')
cfg_m3 = os.path.join(WORKSPACE, 'configs', 'train_m3_data.yaml')

device_flag = "--device cuda" if torch.cuda.is_available() else "--dry_run --device cpu"
cmd_m3 = f"python -u {train_script} --config {cfg_m3} {device_flag} --batch_size 8 --grad_accum_steps 4 --epochs 5"

print(f"[*] Thuc thi lenh: {cmd_m3}")
t0 = time.time()
ret_m3 = os.system(cmd_m3)
elapsed_m3 = time.time() - t0

if ret_m3 != 0:
    raise RuntimeError(f"[X] Huan luyen nhanh M3 that bai voi exit code: {ret_m3}! Vui long kiem tra log ben tren.")
else:
    print(f"[✓] Huan luyen nhanh M3 hoan tat thanh cong trong {elapsed_m3:.2f}s ({elapsed_m3/60:.1f} phut)!")


## 5. Huấn Luyện Nhánh M4: Tích Hợp Distillation & Consistency Loss (5 Epochs)
* **Cấu hình**: `configs/train_m4_distill.yaml`
* **Đặc trưng**: Bổ sung hàm mất mát KL Distillation ($\alpha_{\text{kd}}=0.4$) với Teacher mỏ neo đóng băng và ràng buộc Logits Consistency ($\beta_{\text{cons}}=0.2$) trên cặp ảnh sạch - suy biến.
* **Mục tiêu đối chứng**: Ngăn ngừa hiện tượng quên tri thức trên ảnh sạch phòng lab, duy trì tính bất biến phân loại dưới suy biến nén sâu.


In [ ]:
# ==============================================================================
# 5. HUAN LUYEN NHANH M4 — DATA + DISTILLATION + CONSISTENCY (5 EPOCHS)
# ==============================================================================
cfg_m4 = os.path.join(WORKSPACE, 'configs', 'train_m4_distill.yaml')

device_flag = "--device cuda" if torch.cuda.is_available() else "--dry_run --device cpu"
cmd_m4 = f"python -u {train_script} --config {cfg_m4} {device_flag} --batch_size 8 --grad_accum_steps 4 --epochs 5"

print(f"[*] Thuc thi lenh: {cmd_m4}")
t0 = time.time()
ret_m4 = os.system(cmd_m4)
elapsed_m4 = time.time() - t0

if ret_m4 != 0:
    raise RuntimeError(f"[X] Huan luyen nhanh M4 that bai voi exit code: {ret_m4}! Vui long kiem tra log ben tren.")
else:
    print(f"[✓] Huan luyen nhanh M4 hoan tat thanh cong trong {elapsed_m4:.2f}s ({elapsed_m4/60:.1f} phut)!")


## 6. Huấn Luyện Nhánh M5: M4 + Bộ Điều Hợp LoRA ViT Adapter (5 Epochs)
* **Cấu hình**: `configs/train_m5_lora.yaml`
* **Đặc trưng**: Kế thừa toàn bộ hàm mất mát của M4, đồng thời gắn thêm module LoRA ($r=8, \alpha=16$, 163.840 tham số) vào các tầng MLP của 2 block cuối CLIP ViT-L/14.
* **Mục tiêu đối chứng**: Mở rộng dung lượng biểu diễn vi mô của không gian đặc trưng thị giác mà không làm thay đổi các tầng trích xuất toàn cục đã đóng băng.


In [ ]:
# ==============================================================================
# 6. HUAN LUYEN NHANH M5 — M4 + LORA VIT ADAPTER (5 EPOCHS)
# ==============================================================================
cfg_m5 = os.path.join(WORKSPACE, 'configs', 'train_m5_lora.yaml')

device_flag = "--device cuda" if torch.cuda.is_available() else "--dry_run --device cpu"
cmd_m5 = f"python -u {train_script} --config {cfg_m5} {device_flag} --batch_size 8 --grad_accum_steps 4 --epochs 5"

print(f"[*] Thuc thi lenh: {cmd_m5}")
t0 = time.time()
ret_m5 = os.system(cmd_m5)
elapsed_m5 = time.time() - t0

if ret_m5 != 0:
    raise RuntimeError(f"[X] Huan luyen nhanh M5 that bai voi exit code: {ret_m5}! Vui long kiem tra log ben tren.")
else:
    print(f"[✓] Huan luyen nhanh M5 hoan tat thanh cong trong {elapsed_m5:.2f}s ({elapsed_m5/60:.1f} phut)!")


## 7. Tổng Hợp Bản Ghi Tiến Trình & Trực Quan Hóa Đồ Thị Hội Tụ
* Đọc tệp bản ghi `DATASET/logs/m6_training_history.csv` ghi nhận chi tiết loss và validation metrics qua từng epoch của cả 3 nhánh.
* Vẽ biểu đồ trực quan hóa đường cong hội tụ.
* Đối chiếu sơ bộ với bảng mốc sàn baseline từ Task 6.2.


In [ ]:
# ==============================================================================
# 7. TONG HOP BAN GHI TIEN TRINH VA VE BIEU DO HOI TU
# ==============================================================================
import pandas as pd
import matplotlib.pyplot as plt

log_csv = os.path.join(WORKSPACE, 'DATASET', 'logs', 'm6_training_history.csv')
if os.path.exists(log_csv):
    df = pd.read_csv(log_csv)
    print("[*] Trich doan ban ghi tien trinh huan luyen:")
    display(df.tail(15))

    # Ve bieu do so sanh Loss va ACC
    plt.figure(figsize=(13, 5))

    plt.subplot(1, 2, 1)
    for branch in df['branch'].unique():
        sub = df[df['branch'] == branch]
        plt.plot(sub['epoch'], sub['train_loss'], marker='o', label=f"{branch} Train Loss")
    plt.title("So Sanh Tien Trinh Giam Loss Qua Cac Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    for branch in df['branch'].unique():
        sub = df[df['branch'] == branch]
        plt.plot(sub['epoch'], sub['val_macro_acc'], marker='s', label=f"{branch} Val Macro ACC")
    plt.axhline(y=73.25, color='r', linestyle='--', label='Baseline Moc San (73.25%)')
    plt.title("So Sanh Do Chinh Xac Val Macro ACC")
    plt.xlabel("Epoch")
    plt.ylabel("Macro ACC (%)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    chart_path = os.path.join(WORKSPACE, 'DATASET', 'logs', 'm6_convergence_chart.png')
    plt.savefig(chart_path, dpi=150)
    print(f"[*] Bieu do hoi tu da luu tai: {chart_path}")
    plt.show()
else:
    print(f"[!] Khong tim thay {log_csv}. Vui long kiem tra lai qua trinh huan luyen.")


## 8. Tự Động Sao Lưu Checkpoints & Bản Ghi Sang Google Drive 5TB
* Sao lưu toàn bộ checkpoints (`m3`, `m4`, `m5`) sang Google Drive `/content/drive/MyDrive/Fatformer/checkpoints/`.
* Sao lưu nhật ký huấn luyện `m6_training_history.csv` và biểu đồ hội tụ sang `/content/drive/MyDrive/Fatformer/logs/`.
* In danh mục kiểm toán kích thước và trạng thái tệp an toàn trên Drive.


In [ ]:
# ==============================================================================
# 8. SAO LUU CHECKPOINTS VA NHAT KY SANG GOOGLE DRIVE 5TB
# ==============================================================================
import os
import shutil

src_ckpt = os.path.join(WORKSPACE, 'DATASET', 'checkpoints')
dst_ckpt = '/content/drive/MyDrive/Fatformer/checkpoints'
src_logs = os.path.join(WORKSPACE, 'DATASET', 'logs')
dst_logs = '/content/drive/MyDrive/Fatformer/logs'

print("[*] Dang sao luu checkpoints sang Google Drive...")
if os.path.exists(src_ckpt):
    os.makedirs(dst_ckpt, exist_ok=True)
    for b in ['m3', 'm4', 'm5']:
        b_dir = os.path.join(src_ckpt, b)
        if os.path.exists(b_dir):
            dst_b = os.path.join(dst_ckpt, b)
            os.makedirs(dst_b, exist_ok=True)
            for f in os.listdir(b_dir):
                sp = os.path.join(b_dir, f)
                dp = os.path.join(dst_b, f)
                shutil.copy2(sp, dp)
                print(f"  [+] Da sao luu: {dp} ({os.path.getsize(dp)/(1024**2):.1f} MB)")

print("\n[*] Dang sao luu nhat ky va bieu do sang Google Drive...")
if os.path.exists(src_logs):
    os.makedirs(dst_logs, exist_ok=True)
    for f in os.listdir(src_logs):
        sp = os.path.join(src_logs, f)
        if os.path.isfile(sp):
            dp = os.path.join(dst_logs, f)
            shutil.copy2(sp, dp)
            print(f"  [+] Da sao luu: {dp}")

print("\n[✓] HOAN TAT TOAN BO CHIEN DICH HUAN LUYEN TASK 6.3-C VA SAO LUU AN TOAN!")


## 9. Chuyển Giao Thành Phẩm Cho Task 6.4 & Task 6.5
1. **Bàn giao cho Task 6.5 (Ma Trận Ablation Study M1–M5)**:
   - Toàn bộ kết quả định lượng của 3 nhánh M3, M4, M5 cùng nhánh đối chứng M1 (Plan A) và M2 (Aug only).
2. **Bàn giao cho Task 6.4 (Hiệu Chuẩn Nhiệt Độ & Tuyển Chọn Checkpoint)**:
   - 3 checkpoint đã lưu an toàn trên Google Drive:
     - `/content/drive/MyDrive/Fatformer/checkpoints/m3/crossdomain_model_best.pth`
     - `/content/drive/MyDrive/Fatformer/checkpoints/m4/crossdomain_model_best.pth`
     - `/content/drive/MyDrive/Fatformer/checkpoints/m5/crossdomain_model_best.pth`
   - Sẵn sàng kích hoạt bộ lọc Gate 4 điều kiện trên tập `val_crossdomain` độc lập.
